In [3]:
# Librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Mostrar todas las columnas con pandas
pd.set_option("display.max_columns", None)

In [10]:
# Lectura de set de datos
df = pd.read_csv('03-classification/Telco-Customer-Churn.csv')
# Formatear nombres de columnas y columnas categoricas
df.columns = df.columns.str.lower().str.replace(' ', '_')
cat_columns = df.select_dtypes('string')
for c in cat_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')
df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce').fillna(0)
df['churn'] = (df['churn'] == 'yes').astype(int)
df.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,7590-vhveg,female,0,yes,no,1,no,no_phone_service,dsl,no,yes,no,no,no,no,month-to-month,yes,electronic_check,29.85,29.85,0
1,5575-gnvde,male,0,no,no,34,yes,no,dsl,yes,no,yes,no,no,no,one_year,no,mailed_check,56.95,1889.50,0
2,3668-qpybk,male,0,no,no,2,yes,no,dsl,yes,yes,no,no,no,no,month-to-month,yes,mailed_check,53.85,108.15,1
3,7795-cfocw,male,0,no,no,45,no,no_phone_service,dsl,yes,no,yes,yes,no,no,one_year,no,bank_transfer_(automatic),42.30,1840.75,0
4,9237-hqitu,female,0,no,no,2,yes,no,fiber_optic,no,no,no,no,no,no,month-to-month,yes,electronic_check,70.70,151.65,1


#### **Validation framework**

In [14]:
from sklearn.model_selection import train_test_split

In [17]:
# Separacion en sets de entrenamiento, validation y test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)
print(len(df_train), len(df_val), len(df_test))

4225 1409 1409


In [18]:
# Resetear index
df_train.reset_index(inplace=True, drop=True)
df_val.reset_index(inplace=True, drop=True)
df_test.reset_index(inplace=True, drop=True)

In [19]:
# Separacion de la variable objetivo
y_train = df_train['churn'].values
y_val = df_val['churn'].values
y_test = df_test['churn'].values

In [20]:
# Eliminar variable objetivo del set de datos
del df_train['churn']
del df_val['churn']
del df_test['churn']

#### **EDA**


In [ ]:
# Valores nulos
df_full_train.reset_index(inplace=True, drop=True)
df_full_train.isnull().sum()

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [24]:
# Distribucion de la variable objetivo
df_full_train['churn'].value_counts(normalize=True)

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [25]:
# Global churn rate
global_churn_rate = df_full_train['churn'].mean()
round(global_churn_rate, 2)

np.float64(0.27)

In [ ]:
# Identificacion de variables
cat_columns = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod'
    ]
num_columns = ['tenure', 'monthlycharges', 'totalcharges']

In [27]:
df_full_train[num_columns].describe()

,tenure,monthlycharges,totalcharges
count,5634.000000,5634.000000,5634.000000
mean,32.277955,64.779127,2277.423953
std,24.555211,30.104993,2266.412636
min,0.000000,18.250000,0.000000
25%,9.000000,35.400000,389.137500
50%,29.000000,70.375000,1391.000000
75%,55.000000,89.850000,3787.500000
max,72.000000,118.650000,8684.800000


In [28]:
df_full_train[cat_columns].nunique()

gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

#### **Feature importance**

In [29]:
# Churn rate (groups)
churn_female = df_full_train[df_full_train['gender'] == 'female']['churn'].mean()
churn_male = df_full_train[df_full_train['gender'] == 'male']['churn'].mean()
print(global_churn_rate, churn_female, churn_male)

0.26996805111821087 0.27682403433476394 0.2632135306553911


No existe mayor diferencia --> la variable no discrimina

In [30]:
# Churn rate (groups)
churn_partner_yes = df_full_train[df_full_train['partner'] == 'yes']['churn'].mean()
churn_partner_no = df_full_train[df_full_train['partner'] == 'no']['churn'].mean()
print(global_churn_rate, churn_partner_yes, churn_partner_no)

0.26996805111821087 0.20503330866025166 0.3298090040927694


Si existe diferencia, posible buena variable para predicción

In [31]:
# Difference > 0 --> less likely to churn
# Difference < 0 --> more likely to churn
print(f'Global vs female group      :{global_churn_rate - churn_female}')
print(f'Global vs male group        :{global_churn_rate - churn_male}')
print(f'Global vs partner yes group :{global_churn_rate - churn_partner_yes}')
print(f'Global vs partner no group  :{global_churn_rate - churn_partner_no}')

Global vs female group      :-0.006855983216553063
Global vs male group        :0.006754520462819769
Global vs partner yes group :0.06493474245795922
Global vs partner no group  :-0.05984095297455855


In [33]:
# Risk ratio group/global > 1 more likely to churn
# Risk ratio group/global < 1 less likely to churn
print(f'Global vs female group      :{churn_female / global_churn_rate}')
print(f'Global vs male group        :{churn_male / global_churn_rate}')
print(f'Global vs partner yes group :{churn_partner_yes / global_churn_rate}')
print(f'Global vs partner no group  :{churn_partner_no / global_churn_rate}')

Global vs female group      :1.0253955354648652
Global vs male group        :0.9749802969838747
Global vs partner yes group :0.7594724924338315
Global vs partner no group  :1.2216593879412643


In [34]:
from IPython.display import display
for c in cat_columns:
    print(c)
    df_group = df_full_train.groupby(c).churn.agg(['mean', 'count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk'] = df_group['mean'] / global_churn_rate
    display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121
